# Carnet Log Analysis
Analyse always the most recent log file series of the carnet

In [132]:
import os
from datetime import datetime

log_file_dir = "logs/"
series_opts = ["A-series", "F-series", "H-series"]
# log files are stored in logs/A-series/20260306-094603/run-1-success/zcrr.log

## Helper Functions

In [133]:

# find the most recent log file folder for each series
def find_most_recent_log_file_for_series(series, log_file_dir):
    most_recent_log_folder = None
    most_recent_time = None
    series_dir = os.path.join(log_file_dir, series)
    if not os.path.exists(series_dir):
        print(f"Series directory {series_dir} does not exist.")
        return None
    for log_folder in os.listdir(series_dir):
        log_folder_path = os.path.join(series_dir, log_folder)
        if os.path.isdir(log_folder_path):
            try:
                log_folder_time = datetime.strptime(log_folder, "%Y%m%d-%H%M%S")
                if most_recent_time is None or log_folder_time > most_recent_time:
                    most_recent_time = log_folder_time
                    most_recent_log_folder = log_folder_path
            except ValueError:
                print(f"Skipping folder {log_folder} as it does not match the expected format.")
    return most_recent_log_folder

In [134]:
# find the run folders and split into success and failure runs
def find_run_folders(log_folder):
    all_folders = []
    success_folders = []
    failure_folders = []
    for run_folder in os.listdir(log_folder):
        path = os.path.join(log_folder, run_folder)
        all_folders.append(path)
        if "success" in run_folder:
            success_folders.append(path)
        elif "failure" in run_folder:
            failure_folders.append(path)
    return all_folders, success_folders, failure_folders

In [135]:
stop_phrase = ["Stopping vsomeip application", "Exiting vsomeip application"]
ignore_errors = [
    "send_client_routing_info", 
    "local_uds_client_endpoint_impl::receive_cbk Error: Operation canceled",
    "local_uds_client_endpoint_impl::receive_cbk Error: End of file"
    ]

def analyze_log_file(log_file_path):
    contributed = False
    statistics_active = False
    last_missing = None
    with open(log_file_path, "r") as f:
        lines = f.readlines()
    for line in lines:
        if "contrib" in line.lower():
            contributed = True
        if "statistics_recorder" in line.lower():
            statistics_active = True
            if "check_services_complete" in line.lower() and "has no entries" in line.lower():
                last_missing = line.lower().split("check_services_complete service")[-1].split("has")[0].strip()
        if any(stop in line for stop in stop_phrase):
            break
        if "error" in line.lower():
            if not any(ignore_error in line for ignore_error in ignore_errors):
                print(line.strip())
        if "warning" in line.lower():
            print(line.strip())
    return contributed, statistics_active, last_missing

## Analysis of the most recent log file series

In [136]:
log_folders = dict()
for series in series_opts:
    most_recent_log_folder = find_most_recent_log_file_for_series(series, log_file_dir)
    if most_recent_log_folder:
        print(f"Most recent log folder for {series}: {most_recent_log_folder}")
        log_folders[series] = most_recent_log_folder
    else:
        print(f"No valid log folders found for {series}.")
runs = dict()
for series, log_folder in log_folders.items():
    runs[series] = {"all": [], "success": [], "failure": []}
    all_folders, success_folders, failure_folders = find_run_folders(log_folder)
    runs[series]["all"] = all_folders
    runs[series]["success"] = success_folders
    runs[series]["failure"] = failure_folders
    print(f"{log_folder}: total runs {len(all_folders)} ({len(success_folders)} successful, {len(failure_folders)} failed).")

Most recent log folder for A-series: logs/A-series/20260306-152610
Most recent log folder for F-series: logs/F-series/20260306-152646
Most recent log folder for H-series: logs/H-series/20260306-152748
logs/A-series/20260306-152610: total runs 5 (5 successful, 0 failed).
logs/F-series/20260306-152646: total runs 5 (5 successful, 0 failed).
logs/H-series/20260306-152748: total runs 5 (0 successful, 5 failed).


In [137]:
skip_series = ["A-series", "F-series"] #["A-series", "F-series", "H-series"]
runs_to_analyze = ["run-1"]

for series in runs:
    if series in skip_series:
        print(f"Skipping analysis for {series} as it is marked to be skipped.")
        continue
    print("\n")
    print("-" * 50)
    print(f"Analyzing logs for {series}...")
    for run_folder in runs[series]["all"]:
        if runs_to_analyze is not None and not any(run in run_folder for run in runs_to_analyze):
            continue
        # list all logs in folder
        for log_file in os.listdir(run_folder):
            log_file_path = os.path.join(run_folder, log_file)
            print("-" * 50)
            print(f"Analyzing log file: {log_file_path}")
            contributed, statistics_active, last_missing = analyze_log_file(log_file_path)
            if not statistics_active:
                print("Statistics recorder failed.")
            elif not contributed:
                print(f"Statistics not contributed, missing at least {last_missing}.")

Skipping analysis for A-series as it is marked to be skipped.
Skipping analysis for F-series as it is marked to be skipped.


--------------------------------------------------
Analyzing logs for H-series...
--------------------------------------------------
Analyzing log file: logs/H-series/20260306-152748/run-1-failure/zcrl.log
Statistics not contributed, missing at least 6054.
--------------------------------------------------
Analyzing log file: logs/H-series/20260306-152748/run-1-failure/adas.log
--------------------------------------------------
Analyzing log file: logs/H-series/20260306-152748/run-1-failure/zcrr.log
2026-03-06 15:28:24.021455 [warning] Client 0x32 register timeout! Trying again...
2026-03-06 15:28:24.069775 [warning] Client 0x63 register timeout! Trying again...
2026-03-06 15:28:24.209973 [warning] Client 0x82 register timeout! Trying again...
2026-03-06 15:28:26.844279 [warning] (decimal 154)Client 0x9a request client timeout! Trying again...
2026-03-06 15:28:2